In [ ]:
%load_ext line_profiler

In [ ]:
import numpy as np
newaxis=np.newaxis

from turban.process.shear import ShearProcessing
from turban.utils.util import fast_to_slow_reshape_index

In [ ]:
p = ShearProcessing.from_atomix_netcdf('data/mss/MSS_Baltic.nc', level=1)

In [ ]:
    ii = fast_to_slow_reshape_index(
        p.level2.shear.shape[-1],
        p.cfg.fft_length,
        p.cfg.fft_overlap,
        p.cfg.diss_length,
        p.cfg.diss_overlap,
        p.level1.section_marker,
    )


In [ ]:
def func():
    shear = p.level2.shear.copy()
    Pk = p.level3.Pk.copy()
    k = p.level3.waveno.copy()
    shear_reshape = shear[..., ii]  # reshape to fft length windows
    if True:
        for ishear in range(np.shape(Pk)[0]):    
            for isegment in range(np.shape(Pk)[1]):
                dk = k[ishear,1] - k[ishear,0]
                shear_flat = shear_reshape[ishear,isegment,:,:].flatten()   
                varPk = sum(Pk[ishear,isegment,:]) * dk
                varscale = np.var(shear_flat) / varPk
                Pk[ishear,isegment,:] *= varscale


%lprun -f func func()

In [ ]:
%%time
func()

In [ ]:
def func():
    shear = p.level2.shear.copy()
    Pk = p.level3.Pk.copy()
    Pk = np.ascontiguousarray(Pk)
    k = p.level3.waveno.copy()
    
    dk = k[..., 1] - k[..., 0]
    varPk = Pk.sum(axis=-1) * dk[newaxis, :]
    sr = shear[:, ii]
    # srf = sr.reshape(sr.shape[:-2] + (sr.shape[-2]*sr.shape[-1],)) # flatten last two axes
    srf = np.ascontiguousarray(sr.reshape(sr.shape[:-2] + (sr.shape[-2]*sr.shape[-1],)))
    varscale = np.var(srf, axis=-1) 
    Pk *= varscale[..., newaxis] / varPk[..., newaxis]

%lprun -f func func()

In [ ]:
%%time
func()

In [ ]:
shear = p.level2.shear.copy()

sr = shear[:, ii]
srf = sr.reshape(sr.shape[:-2] + (sr.shape[-2]*sr.shape[-1],)) # flatten last two axes

# srf = np.asfortranarray(srf)
srf = np.ascontiguousarray(srf)

In [ ]:
%%time
for i in range(srf.shape[-1]):
    _ = np.var(srf[..., i], axis=-1)

In [ ]:
%%time
for i in range(srf.shape[-1]):
    _ = np.var(srf[..., i])

In [ ]:
%%time
_ = np.var(srf, axis=0)

In [ ]:
%%time
_ = np.var(srf, axis=1)

In [ ]:
%%time
_ = np.var(srf)

In [ ]:
a = np.random.random((2000, 2000))
# a = np.ascontiguousarray(a)

In [ ]:
%%time
for i in range(a.shape[-1]):
    _ = np.sum(a[:, i])

In [ ]:
%%time
_ = np.sum(a, axis=1)

In [ ]:
%%time
_ = np.sum(a, axis=0)

In [ ]:
%%time
_ = np.sum(a)

In [ ]:
%%time
_ = np.var(a, axis=1)

In [ ]:
%%time
_ = np.var(a, axis=0)

In [ ]:
%%time
_ = np.var(a)